# Libya Land-Cover Assistant — English chat, no paid API

**Run the two cells below in order. No code edits, API key, payment card or Colab Pro subscription are required by this notebook.**

1. Click ▶ on **Set up** and wait for completion.
2. Click ▶ on **Open assistant**. The first model download can take several minutes.
3. In the assistant, click **Upload result ZIP** and select your previous pilot/mapping backup. Then ask: **“What is the status of my result?”**

For a new map, use **Prepare map** or ask in English. Connect Earth Engine using the button before running new work. You need your existing authorized Earth Engine project and asset access. The configured example is Tripoli; public GitHub code does not grant access to its assets.

The chat model runs inside your own Colab runtime. Your laptop only displays the notebook. CPU is supported; an available GPU can be used automatically. Free Colab resources and session duration are limited and not guaranteed. Keep this notebook open during mapping and download your backup before leaving. This is an interactive notebook, not an always-on public website.


In [ ]:
# @title 1. Set up — run once in a fresh runtime
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
SERVICE_COMMIT = "a8a77c273c70e9fd184e13d6b909e448a7c2cc74"
print("Step 1/2: preparing the notebook. Please wait…", flush=True)
try:
    torch_version = tuple(int(n) for n in version("torch").split("+")[0].split(".")[:2])
except PackageNotFoundError:
    torch_version = (0, 0)
if torch_version < (2, 6):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
        "torch>=2.6,<3", "--index-url", "https://download.pytorch.org/whl/cpu"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
    "libya-landcover-assistant[local-chat] @ git+https://github.com/hamedsabzchi/libya-automated-land-cover-mapping.git@" + SERVICE_COMMIT])
import libya_landcover
if libya_landcover.__version__ != "0.7.0":
    raise RuntimeError("An older assistant is already loaded. Choose Runtime > Disconnect and delete runtime, then run these two cells again.")
print("Setup complete. Now run Open assistant below.")


In [ ]:
# @title 2. Open assistant — wait for “English chat ready”
from pathlib import Path
from google.colab import files, output
from libya_landcover.assistant import Assistant, TRIPOLI
from libya_landcover.job_store import JobStore
from libya_landcover.local_language import LocalParser
from libya_landcover.chat_language import GuidedParser
from libya_landcover.chat_ui import display_chat

if "SERVICE_COMMIT" not in globals():
    raise RuntimeError("Run Set up first.")
output.enable_custom_widget_manager()
print("Step 2/2: loading English chat. No API key or payment is needed.", flush=True)
try:
    if "local_parser" not in globals() or not isinstance(local_parser, LocalParser):
        local_parser = LocalParser.load(cache_dir="/content/landcover-chat-model")
    parser = local_parser
except Exception as exc:
    print("The local language model could not load (" + type(exc).__name__ + ").")
    print("This session will use guided buttons/commands only. To retry English chat, run this cell again.")
    parser = GuidedParser()

store = JobStore(Path("/content/assistant-store"))
assistant = Assistant(store, engine_commit=SERVICE_COMMIT, profile=dict(TRIPOLI), parser=parser)

def connect_earth_engine():
    import ee
    ee.Authenticate()
    ee.Initialize(project=assistant.profile["project"])

ui = display_chat(assistant, download=files.download, connect=connect_earth_engine)


## What to try

- **“What is the status of my result?”** after uploading your ZIP.
- **“Where can I download the output?”** to see an available Drive link.
- **“Prepare a land-cover map for Tripoli for 2024.”** to review settings.
- **“Change the year to 2023.”** to revise the pending request.

A message never starts mapping by itself. Review the settings, connect Earth Engine if needed, then click **Run reviewed request**. **Prepare pilot** exports one small window from an existing completed mapping after its own preview and confirmation.

**Download backup** saves the selected completed mapping or pilot for the next session. It is not the TIFF; obtain raster files from the Drive link. After a runtime reset, run the two cells and upload your latest backup again. An active Python mapping worker cannot be restored from a completed-result ZIP.

This version can prepare maps and explain recorded jobs/results. It does not yet deliver a complete tiled region or calculate class-area/change statistics. Small local models can misunderstand requests: always check the displayed boundary, training data and year. No claim about classifier accuracy is made by the assistant.

There is no paid inference fallback and no external chat server/tunnel. If the model is unavailable, the interface explicitly says **Guided commands** and its buttons remain available. The English free-text mode still needs a first user-side Colab check; automated tests alone cannot guarantee all Colab environments.
